In [ ]:
from ddi.data import build_human
from ddi.manifest import load_dataset

train, dev, val = build_human()
v14, _ = load_dataset("20260807-123340-ff79db")

In [ ]:
import re, statistics
from collections import Counter

ROLE = re.compile(
    r"\b(previous(ly)?|earlier|discontinu\w+|stopped|withdraw\w+|"
    r"subsequent(ly)?|thereafter|later|after the other\w*|"
    r"comparator|separately|comparison|"
    r"alternative|unsuitable|first choice|instead|"
    r"not given|not permitted|excluded|withheld|not administered)\b", re.I)

def near_marker(text, chars=60):
    """text within `chars` either side of each entity span"""
    out = []
    for k in (1, 2):
        m = re.search(rf"\[E{k}\].*?\[/E{k}\]", text, re.S)
        if m:
            out.append(text[max(0, m.start()-chars):m.end()+chars])
    return " ".join(out)

def role_adjacency(instances, name):
    c = Counter()
    for r in instances:
        hit = bool(ROLE.search(near_marker(r["text"])))
        c[(r["label"] != "NONE", hit)] += 1
    pos_rate = c[(True, True)] / max(c[(True, True)] + c[(True, False)], 1)
    neg_rate = c[(False, True)] / max(c[(False, True)] + c[(False, False)], 1)
    print(f"{name:8s} role language near marker: POS {pos_rate:.3f}  NONE {neg_rate:.3f}"
          f"  ratio {neg_rate / max(pos_rate, 1e-9):.2f}")

role_adjacency(v14, "v14")
role_adjacency(train, "human")

In [ ]:
# P_ROLE=0.2 smoke test
# Change ONE thing in ddi/prompt.py before running: P_ROLE = 0.55 -> 0.2
# Everything else stays as it was for v14-full-2, so the comparison is single-variable.
import os
from openai import OpenAI
import importlib, json, re
from collections import Counter

import ddi.prompt, ddi.resolve, ddi.gates, ddi.synth
for m in (ddi.synth, ddi.prompt, ddi.resolve, ddi.gates):
    importlib.reload(m)

from ddi.data import build_human
from ddi.vocab import build_vocab
from ddi.prompt import make_v14_specs, make_v14_sample_fn, v14_fingerprint, P_ROLE
from ddi.resolve import v14_sample_to_instances, generation_records
from ddi.synth import generate_raw, build_dataset_from_raw, RAW
from ddi.manifest import load_dataset
from ddi import gates

GEN = "v14-lowrole-check"
V14_ID = "20260807-123340-ff79db"
MODEL, API, EFFORT = "gpt-oss-120b", "responses", "low"

client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=5, timeout=60.0)

print(f"P_ROLE = {P_ROLE}  (must be 0.2)")
print(f"prompt sha {v14_fingerprint()}")

vocab = build_vocab()
train, dev, val = build_human()
v14, _ = load_dataset(V14_ID)


# ---- generate -------------------------------------------------------------
specs = make_v14_specs(300, vocab=vocab, seed=0)

n_roled = sum(len(s["roles"]) for s in specs)
n_np = sum(len([e for e in s["entities"]
                if e["key"] not in {k for a in s["asserts"] for k in a["between"]}])
           for s in specs)
print(f"specs: {n_roled}/{n_np} non-participants carry a role "
      f"({n_roled / max(n_np, 1):.2f})")

generate_raw(specs, make_v14_sample_fn(client, model=MODEL, reasoning_effort=EFFORT,
                                       api=API),
             gen_id=GEN, max_workers=16)

did, stats = build_dataset_from_raw(
    GEN, resolver=v14_sample_to_instances, mode="markers",
    generator={"prompt_sha": v14_fingerprint(), "p_role": P_ROLE,
               "model": MODEL, "reasoning_effort": EFFORT,
               "note": "P_ROLE ablation against v14-full-2"},
    vocab_source=vocab.fingerprint(), seed=0)
print(stats["reject_reasons"])
inst, _ = load_dataset(did)


# ---- manipulation check: did the cue actually move? -----------------------
ROLE = re.compile(
    r"\b(previous(ly)?|earlier|discontinu\w+|stopped|withdraw\w+|"
    r"subsequent(ly)?|thereafter|later|after the other\w*|"
    r"comparator|separately|comparison|"
    r"alternative|unsuitable|first choice|instead|"
    r"not given|not permitted|excluded|withheld|not administered)\b", re.I)

def near_marker(text, chars=60):
    out = []
    for k in (1, 2):
        m = re.search(rf"\[E{k}\].*?\[/E{k}\]", text, re.S)
        if m:
            out.append(text[max(0, m.start() - chars):m.end() + chars])
    return " ".join(out)

def role_adjacency(instances, name):
    c = Counter()
    for r in instances:
        c[(r["label"] != "NONE", bool(ROLE.search(near_marker(r["text"]))))] += 1
    pos = c[(True, True)] / max(c[(True, True)] + c[(True, False)], 1)
    neg = c[(False, True)] / max(c[(False, True)] + c[(False, False)], 1)
    print(f"{name:12s} POS {pos:.3f}  NONE {neg:.3f}  ratio {neg / max(pos, 1e-9):.2f}")
    return neg

print("\nrole language near marker")
role_adjacency(train, "human")          # 0.018 / 0.005, ratio 0.27
role_adjacency(v14, "v14 (0.55)")       # 0.228 / 0.468, ratio 2.05
new_neg = role_adjacency(inst, "v14 (0.20)")
print(f"\nNONE adjacency {0.468:.3f} -> {new_neg:.3f}; corpus is 0.005")


# ---- side effects: the other gates must not have moved -------------------
gates.report(inst, records=generation_records(GEN), strict=False)


# ---- read the zero-assert sentences --------------------------------------
# most non-participants now have no role, so the scene has to carry them.
# the failure to look for is bare lists returning.
shown = 0
for line in (RAW / f"{GEN}.jsonl").read_text().splitlines():
    r = json.loads(line)
    if r.get("error") or r["spec"]["asserts"] or shown >= 10:
        continue
    print(f"[{len(r['spec']['entities'])} ents, {len(r['spec']['roles'])} roles] "
          f"{r['sample']['sentence']}")
    shown += 1

In [ ]:
import os
from openai import OpenAI

client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=5, timeout=60.0)

In [ ]:
# P_ROLE = 0.2 ablation: full run, train, compare against v14-full-2
#
# Single variable against v14-full-2, with one caveat: P_ROLE also shifted the
# hard-negative rate (0.499 -> 0.612 in smoke), so composition is not perfectly held.
# Record that in the ablation row rather than claiming a clean isolation.
#
# Prerequisite: ddi/prompt.py has P_ROLE = 0.2 and nothing else changed since
# v14-full-2. Commit before running; manifests are landing DIRTY.

# ===========================================================================
# Cell 1: generate. ~30 min at 3.5 req/s. Run under tmux if you want to detach.
# ===========================================================================
import importlib, json, re, statistics, random
from collections import Counter
import pandas as pd

import ddi.prompt, ddi.resolve, ddi.gates, ddi.synth, ddi.train
for m in (ddi.synth, ddi.prompt, ddi.resolve, ddi.gates, ddi.train):
    importlib.reload(m)

from ddi.data import build_human, load_brat_docs
from ddi.vocab import build_vocab
from ddi.prompt import make_v14_specs, make_v14_sample_fn, v14_fingerprint, P_ROLE
from ddi.resolve import v14_sample_to_instances, generation_records
from ddi.synth import generate_raw, build_dataset_from_raw, RAW
from ddi.manifest import load_dataset
from ddi.train import train_and_eval
from ddi import gates

GEN = "v14-lowrole-full"
V14_ID = "20260807-123340-ff79db"
V13_ID = "<the ~19k v13 dataset id>"
MODEL, API, EFFORT = "gpt-oss-120b", "responses", "low"

assert P_ROLE == 0.2, f"P_ROLE is {P_ROLE}, expected 0.2"
print(f"prompt sha {v14_fingerprint()}")

vocab = build_vocab()
specs = make_v14_specs(6000, vocab=vocab, seed=0)
generate_raw(specs, make_v14_sample_fn(client, model=MODEL, reasoning_effort=EFFORT,
                                       api=API),
             gen_id=GEN, max_workers=16)

low_id, stats = build_dataset_from_raw(
    GEN, resolver=v14_sample_to_instances, mode="markers",
    generator={"prompt_sha": v14_fingerprint(), "p_role": P_ROLE,
               "model": MODEL, "reasoning_effort": EFFORT, "composition": "prior",
               "n_positives_by_k": "hand-tuned to decorrelate positive rate from "
                                   "entity count; not a corpus measurement",
               "note": "P_ROLE ablation against v14-full-2"},
    vocab_source=vocab.fingerprint(), seed=0,
    notes="v14 with P_ROLE=0.2, 6000 specs")
print(low_id, stats["reject_reasons"])



In [ ]:
import importlib, json, re, statistics, random
from collections import Counter
import pandas as pd

import ddi.prompt, ddi.resolve, ddi.gates, ddi.synth, ddi.train
for m in (ddi.synth, ddi.prompt, ddi.resolve, ddi.gates, ddi.train):
    importlib.reload(m)

from ddi.data import build_human, load_brat_docs
from ddi.vocab import build_vocab
from ddi.prompt import make_v14_specs, make_v14_sample_fn, v14_fingerprint, P_ROLE
from ddi.resolve import v14_sample_to_instances, generation_records
from ddi.synth import generate_raw, build_dataset_from_raw, RAW
from ddi.manifest import load_dataset
from ddi.train import train_and_eval
from ddi import gates

# ===========================================================================
# Cell 2: manipulation check + gates at full scale.
# The probe is scale-dependent, so this number is not comparable to the smoke run.
# ===========================================================================

GEN = "v14-lowrole-full"
V14_ID = "20260807-123340-ff79db"
low_id = "20260812-015921-2b80ca"

train, dev, val = build_human()
v14, _ = load_dataset(V14_ID)
low, _ = load_dataset(low_id)
print(f"human {len(train)}, v14 {len(v14)}, low {len(low)} instances")

ROLE = re.compile(
    r"\b(previous(ly)?|earlier|discontinu\w+|stopped|withdraw\w+|"
    r"subsequent(ly)?|thereafter|later|after the other\w*|"
    r"comparator|separately|comparison|"
    r"alternative|unsuitable|first choice|instead|"
    r"not given|not permitted|excluded|withheld|not administered)\b", re.I)

def near_marker(text, chars=60):
    out = []
    for k in (1, 2):
        m = re.search(rf"\[E{k}\].*?\[/E{k}\]", text, re.S)
        if m:
            out.append(text[max(0, m.start() - chars):m.end() + chars])
    return " ".join(out)

def role_adjacency(instances, name):
    c = Counter()
    for r in instances:
        c[(r["label"] != "NONE", bool(ROLE.search(near_marker(r["text"]))))] += 1
    pos = c[(True, True)] / max(c[(True, True)] + c[(True, False)], 1)
    neg = c[(False, True)] / max(c[(False, True)] + c[(False, False)], 1)
    print(f"{name:12s} POS {pos:.3f}  NONE {neg:.3f}  ratio {neg / max(pos, 1e-9):.2f}")

print("\nrole language near marker")
role_adjacency(train, "human")
role_adjacency(v14, "v14 (0.55)")
role_adjacency(low, "low (0.20)")


In [ ]:

GEN = "v14-lowrole-full"
gates.report(low, records=generation_records(GEN), strict=False)



In [ ]:

# ===========================================================================
# Cell 3: train. 3 seeds, ~1 min each. v14 numbers are already known
# (F1 0.379, P 0.302, R 0.511) so only the new arm needs running.
# ===========================================================================
BASE = {"model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256,
        "neg_ratio": None, "render_mode": "markers"}

rows, preds = [], {}
for seed in [0]:
    cfg = {**BASE, "seed": seed, "dataset": "v14-lowrole"}
    m, p = train_and_eval(cfg, low, dev, return_preds=True)
    if seed == 0:
        preds["low"] = p
    rows.append({"arm": "v14-lowrole", "seed": seed,
                 "f1": m["micro_f1_pos"], "p": m["micro_p_pos"], "r": m["micro_r_pos"],
                 "f1_DrugBank": m.get("micro_f1_pos_DrugBank"),
                 "f1_MedLine": m.get("micro_f1_pos_MedLine")})
    print(f"lowrole seed={seed}  f1={m['micro_f1_pos']:.3f}  "
          f"p={m['micro_p_pos']:.3f}  r={m['micro_r_pos']:.3f}")

df = pd.DataFrame(rows)
print(df[["f1", "p", "r", "f1_DrugBank", "f1_MedLine"]].agg(["mean", "std"]))

print("\nreference (3 seeds, same dev):")
print("  human        F1 0.800  P 0.768  R 0.835")
print("  v13          F1 0.280  P 0.182  R 0.610")
print("  v14 (0.55)   F1 0.379  P 0.302  R 0.511")
print("\nprecision is the target. seed sd ~0.013, so P > ~0.34 is real.")
print("if P rises and R falls proportionally, that is a trade, not a fix.")



In [ ]:
# MECHANISM diagnosis
#
# v14 per-class precision: MECHANISM 0.23, EFFECT 0.42, ADVISE 0.30, INT 0.22.
# Both MECHANISM and EFFECT come from the same machinery, so the asymmetry is
# informative: something about how MECHANISM content is generated is worse.
#
# Everything here reads the TRAIN split and dev predictions from a model already
# trained. Reading dev predictions to diagnose your own generator is ordinary model
# development; the line is the test set, which stays sealed.
#
# Assumes cell 3 has run and preds["v14"] exists. If you only have the lowrole
# predictions in the kernel, retrain the v14 arm first:
#
#   v14, _ = load_dataset(V14_ID)
#   _, preds_v14 = train_and_eval({**BASE, "seed": 0, "dataset": "v14"},
#                                 v14, dev, return_preds=True)

import re, random
from collections import Counter
import pandas as pd
from sklearn.metrics import confusion_matrix

from ddi.data import build_human, load_brat_docs, make_sentence_level

train, dev, val = build_human()
P = preds["v14"]                       # rename if your dict key differs
LABELS = ["NONE", "MECHANISM", "EFFECT", "ADVISE", "INT"]


# ===========================================================================
# 1. Confusion matrix.
# If MECHANISM false positives are mostly true NONE, the problem is over-firing
# and richer content will not help much. If they are mostly true EFFECT, the
# scope separation is failing and sharper boundary content is the fix.
# ===========================================================================
gold = [r["label"] for r in dev]
cm = confusion_matrix(gold, P, labels=LABELS)
print("rows = gold, cols = predicted\n")
print(pd.DataFrame(cm, index=LABELS, columns=LABELS))

print("\nwhat each predicted class actually was:")
for pred in LABELS[1:]:
    src = Counter(g for g, p in zip(gold, P) if p == pred)
    n = sum(src.values())
    if n:
        print(f"  pred {pred:<10} n={n:<5} " +
              "  ".join(f"{k} {v / n:.2f}" for k, v in src.most_common()))


# ===========================================================================
# 2. Read the MECHANISM false positives.
# These tell you what v14's model thinks a mechanism looks like.
# ===========================================================================
def strip(t):
    return re.sub(r"\[/?E[12]\]", "", t)

fps = [(d["text"], d["label"]) for d, p in zip(dev, P)
       if p == "MECHANISM" and d["label"] != "MECHANISM"]
print(f"\n{len(fps)} MECHANISM false positives; showing 20 by gold class\n")
for glab in ["NONE", "EFFECT", "ADVISE", "INT"]:
    sub = [t for t, g in fps if g == glab][:5]
    if not sub:
        continue
    print(f"--- gold {glab} ---")
    for t in sub:
        print(f"  {t}\n")


# ===========================================================================
# 3. And the MECHANISM false negatives: real mechanism sentences it missed.
# ===========================================================================
fns = [d["text"] for d, p in zip(dev, P)
       if d["label"] == "MECHANISM" and p != "MECHANISM"]
print(f"\n{len(fns)} MECHANISM false negatives; showing 10\n")
for t in fns[:10]:
    print(f"  {t}\n")


# ===========================================================================
# 4. Real MECHANISM sentences from the TRAIN split.
# This is the material for redesigning the content slots. Paste the output.
# ===========================================================================
def doc_keys(instances):
    keys = set()
    for r in instances:
        _, register, rest = r["sent_id"].split(":", 2)
        keys.add((register, rest.rsplit(":", 1)[0]))
    return keys

train_keys = doc_keys(train)
train_docs = [d for d in load_brat_docs("Train")
              if (d.register, d.doc_id) in train_keys]

mech = []
for doc in train_docs:
    for sent in make_sentence_level(doc):
        ids = {e.id: e for e in sent.entities}
        for rel in sent.relations:
            if rel.type != "MECHANISM":
                continue
            a1, a2 = rel.arguments["Arg1"], rel.arguments["Arg2"]
            if a1 in ids and a2 in ids:
                mech.append((sent.register, ids[a1].text, ids[a2].text, sent.text))

print(f"\n{len(mech)} MECHANISM relations in the train split")
rng = random.Random(0)
for reg in ("DrugBank", "MedLine"):
    sub = [m for m in mech if m[0] == reg]
    rng.shuffle(sub)
    print(f"\n===== {reg} ({len(sub)}) =====")
    for _, a, b, txt in sub[:25]:
        print(f"[{a} ~ {b}]\n  {txt}\n")


# ===========================================================================
# 5. What vocabulary real MECHANISM sentences use that v14 does not.
# v14 sites: hepatic metabolism, CYP-mediated metabolism, oxidative metabolism,
# liver enzyme activity, first-pass metabolism, gastrointestinal absorption,
# uptake from the gut, oral absorption, plasma protein binding, binding to serum
# albumin, protein-bound fraction, renal clearance, excretion by the kidney,
# tubular secretion, urinary elimination, P-glycoprotein transport, efflux
# transport, carrier-mediated transport, intestinal transport, gastric pH,
# stomach acidity, gastric acid secretion. Plus "higher" / "lower".
# ===========================================================================
texts = [t for _, _, _, t in mech]

PATTERNS = {
    "named CYP enzyme": r"\bCYP\s?[0-9][A-Z][0-9]*\b",
    "inhibit*": r"\binhibit\w*",
    "induc*": r"\binduc\w*",
    "metaboli*": r"\bmetaboli\w*",
    "clearance": r"\bclearance\b",
    "AUC / Cmax / half-life": r"\b(AUC|Cmax|C max|half-?life|t1/2)\b",
    "plasma concentration": r"\bplasma (concentration|level)",
    "serum concentration": r"\bserum (concentration|level)",
    "absorption": r"\babsorpt\w*",
    "protein binding": r"\bprotein[- ]bind\w*",
    "renal / excretion": r"\b(renal|excret\w*|urinary)",
    "transporter / P-gp": r"\b(P-?gp|P-?glycoprotein|transporter)",
    "enzyme (generic)": r"\benzyme\w*",
    "substrate": r"\bsubstrate\b",
    "bioavailability": r"\bbioavailab\w*",
    "steady state": r"\bsteady[- ]state\b",
    "percentage figure": r"\d+\s?%",
    "fold change": r"\d+\s?-?\s?fold",
}
print(f"\nvocabulary in {len(texts)} real MECHANISM sentences")
for name, pat in PATTERNS.items():
    hits = sum(bool(re.search(pat, t, re.I)) for t in texts)
    print(f"  {name:<24} {hits:>4}  {hits / max(len(texts), 1):.3f}")

# specific enzymes, for building the slot pool
enz = Counter(m.upper().replace(" ", "")
              for t in texts for m in re.findall(r"\bCYP\s?[0-9][A-Z][0-9]*\b", t, re.I))
print("\nnamed enzymes:", enz.most_common(20))